In [ ]:
# %% [markdown]
# # 🧠 Notebook 02: Model Architecture Prototyping
# **Objective:** Initialize models and sanity check tensor shapes.

# %%
import torch
import sys
import os
from pathlib import Path

sys.path.append(str(Path(os.getcwd()).parent))
from src.models.gnn_encoder import GNNEncoder
from src.models.diffusion import BioDiffusion

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Running on: {device}")

# %% [markdown]
# ## 1. Test GNN Encoder
# 

# %%
# Simulate a batch of 4 proteins
batch_size = 4
num_nodes = 50
features = 9

# Fake protein graph data
x = torch.randn(num_nodes * batch_size, features).to(device)
edge_index = torch.randint(0, num_nodes * batch_size, (2, num_nodes * 2)).to(device)
batch_vec = torch.repeat_interleave(torch.arange(batch_size), num_nodes).to(device)

# Initialize
encoder = GNNEncoder(num_node_features=features, embedding_dim=128).to(device)

# Forward pass
from torch_geometric.data import Data, Batch
data_list = [Data(x=x[i*num_nodes:(i+1)*num_nodes], edge_index=edge_index) for i in range(batch_size)]
batch = Batch.from_data_list(data_list).to(device)

context_embedding = encoder(batch)
print(f"Context Embedding Shape: {context_embedding.shape} (Should be [4, 128])")

# %% [markdown]
# ## 2. Test Diffusion Denoiser
# 

[Image of Diffusion Process]


# %%
# Fake Ligand inputs (Noisy coordinates)
# [Batch, Atoms, XYZ]
noisy_ligand = torch.randn(batch_size, 20, 3).to(device)
time_step = torch.randint(0, 1000, (batch_size,)).to(device)

diffusion = BioDiffusion(hidden_dim=128).to(device)

# Predict noise
predicted_noise = diffusion.denoise_net(noisy_ligand, time_step, context_embedding)
print(f"Predicted Noise Shape: {predicted_noise.shape} (Should match input [4, 20, 3])")